# Step 2 — Government Quality Score (MIPS)

**Business question:** Given a list of Massachusetts provider NPIs, what is each provider's CMS MIPS quality score?

**Logic:** Pull QPP/MIPS final scores, enrich with Care Compare and Part B utilization data, and produce a clean per-NPI table. Business logic (three-situation classification) will be designed after exploring this data.

**Structural ceiling:** All CMS data is Medicare fee-for-service. Providers with mostly commercial patients will look thin in this data. This is a known limitation, not something we can engineer around.

## Step 1 — Load Data

**What this does:** Load the four datasets needed for the MIPS quality score pipeline:
1. **QPP/MIPS Final Scores** — CMS's official quality assessment per provider
2. **Care Compare** — clinician-level enrichment (telehealth, affiliations)
3. **Part B Utilization** — billing volume and specialty data
4. **NPPES** — provider registry filtered to Massachusetts (our master NPI list)

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import os
import glob

# --- File detection ---
# QPP/MIPS: try known filename patterns
qpp_candidates = glob.glob("2_datasets/ec_score*.csv") + glob.glob("2_datasets/MIPS*2023*.csv") + glob.glob("2_datasets/*qpp*.csv") + glob.glob("2_datasets/*QPP*.csv")
if not qpp_candidates:
    raise FileNotFoundError(
        "QPP/MIPS file not found. Expected 'ec_score_file.csv' or similar in 2_datasets/. "
        "Download from https://data.cms.gov/provider-data/topics/doctors-clinicians (QPP section)"
    )
qpp_file = qpp_candidates[0]
print(f"Using QPP file: {qpp_file}")

In [ ]:
# Load QPP/MIPS final scores
qpp = pd.read_csv(qpp_file, dtype=str)
print(f"QPP/MIPS: {qpp.shape[0]:,} rows, {qpp.shape[1]} columns")
print(f"Columns: {list(qpp.columns)}")

# Sanity check: expect ~800k-1M rows nationally
if qpp.shape[0] < 500_000:
    print(f"⚠️ WARNING: Only {qpp.shape[0]:,} rows. Expected ~800k-1M. Check if this is the right file/year.")
elif qpp.shape[0] > 2_000_000:
    print(f"⚠️ WARNING: {qpp.shape[0]:,} rows is unusually high. May contain multiple years.")
else:
    print(f"✓ Row count looks reasonable for national MIPS data")

In [ ]:
# Load Care Compare clinician data
cc_candidates = glob.glob("2_datasets/DAC_National*.csv") + glob.glob("2_datasets/*care_compare*.csv")
if not cc_candidates:
    raise FileNotFoundError(
        "Care Compare file not found. Expected 'DAC_NationalDownloadableFile.csv' in 2_datasets/. "
        "Download from https://data.cms.gov/provider-data/topics/doctors-clinicians"
    )
cc_file = cc_candidates[0]
print(f"Using Care Compare file: {cc_file}")

care_compare = pd.read_csv(cc_file, dtype=str)
print(f"Care Compare: {care_compare.shape[0]:,} rows, {care_compare.shape[1]} columns")
print(f"Columns: {list(care_compare.columns)}")

In [ ]:
# Load Medicare Part B utilization
partb_candidates = (
    glob.glob("2_datasets/MUP_PHY*Prov_Svc*.csv")
    + glob.glob("2_datasets/Medicare_Physician*Provider_and_Service*.csv")
)
if not partb_candidates:
    raise FileNotFoundError(
        "Part B file not found. Expected 'MUP_PHY_R25_P05_V20_D23_Prov_Svc.csv' in 2_datasets/. "
        "Download from https://data.cms.gov/provider-summary-by-type-of-service/"
    )
partb_file = partb_candidates[0]
print(f"Using Part B file: {partb_file}")
print(f"File size: {os.path.getsize(partb_file) / 1e9:.1f} GB")

# Load with only needed columns to save memory (~3GB file)
partb_usecols = [
    "Rndrng_NPI", "Rndrng_Prvdr_Last_Org_Name", "Rndrng_Prvdr_First_Name",
    "Rndrng_Prvdr_Type", "Rndrng_Prvdr_State_Abrvtn",
    "HCPCS_Cd", "HCPCS_Desc", "Place_Of_Srvc",
    "Tot_Benes", "Tot_Srvcs", "Tot_Bene_Day_Srvcs",
    "Avg_Sbmtd_Chrg", "Avg_Mdcr_Alowd_Amt", "Avg_Mdcr_Pymt_Amt"
]
partb = pd.read_csv(partb_file, dtype=str, usecols=partb_usecols)
print(f"Part B: {partb.shape[0]:,} rows, {partb.shape[1]} columns")

# Sanity check
if partb.shape[0] < 5_000_000:
    print(f"⚠️ WARNING: Only {partb.shape[0]:,} rows. National Part B by-service typically has 9M+ rows.")
else:
    print(f"✓ Row count looks reasonable for national Part B data")

In [ ]:
# Load NPPES — chunked, filter to MA on the fly
nppes_candidates = glob.glob("2_datasets/NPPES Data/npidata_pfile_*.csv")
# Exclude the fileheader file
nppes_candidates = [f for f in nppes_candidates if "fileheader" not in f]
if not nppes_candidates:
    raise FileNotFoundError(
        "NPPES file not found. Expected 'npidata_pfile_*.csv' in 2_datasets/NPPES Data/. "
        "Download from https://download.cms.gov/nppes/NPI_Files.html and extract the ZIP."
    )
nppes_file = nppes_candidates[0]
print(f"Using NPPES file: {nppes_file}")
print(f"File size: {os.path.getsize(nppes_file) / 1e9:.1f} GB")

# Key columns to keep (minimizes memory — full file has 330 columns)
nppes_cols = [
    "NPI",
    "Entity Type Code",
    "Provider Last Name (Legal Name)",
    "Provider First Name",
    "Provider Credential Text",
    "Provider Business Practice Location Address State Name",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address Postal Code",
    "Healthcare Provider Taxonomy Code_1",
    "Provider Enumeration Date",
    "NPI Deactivation Date",
]

# Chunked load, filter to MA
print("Loading NPPES and filtering to MA (this takes a few minutes)...")
ma_chunks = []
total_rows = 0
for chunk in pd.read_csv(nppes_file, dtype=str, usecols=nppes_cols, chunksize=50_000):
    total_rows += len(chunk)
    ma_chunk = chunk[
        chunk["Provider Business Practice Location Address State Name"]
        .str.upper().str.strip()
        .isin(["MA", "MASSACHUSETTS"])
    ]
    if len(ma_chunk) > 0:
        ma_chunks.append(ma_chunk)

nppes_ma = pd.concat(ma_chunks, ignore_index=True)
print(f"NPPES total rows processed: {total_rows:,}")
print(f"NPPES MA providers: {nppes_ma.shape[0]:,}")

# Sanity check: expect ~80k-120k MA NPIs
if nppes_ma.shape[0] < 30_000:
    print(f"⚠️ WARNING: Only {nppes_ma.shape[0]:,} MA NPIs. Expected ~80k-120k. Check filter column.")
elif nppes_ma.shape[0] > 200_000:
    print(f"⚠️ WARNING: {nppes_ma.shape[0]:,} MA NPIs is unusually high. Filter may not be working.")
else:
    print(f"✓ MA NPI count looks reasonable")

## Step 2 — Exploratory Data Analysis

**What this does:** Before cleaning or joining anything, understand what we're working with. Schema, shape, null rates, distributions, and key charts for each dataset.

## Step 3 — Data Preprocessing & Cleaning

**What this does:** Harmonize NPIs, filter to MA, aggregate Part B to per-NPI, deduplicate, and build the master table via left joins.

## Output Schema

One row per MA NPI. MIPS scores where available, Part B utilization summary, Care Compare enrichment. This table is the input for the business logic brainstorm (Step 4, to be designed).